# Verification: Symmetric Bivalent Hapten-Receptor Cross-Linking (No Rings)

Compares NFsim against Perelson and DeLisi (1980) kinetic ODEs (Eqs. 1–2)
and exact per-site equilibrium theory.

**Combinatorial mapping:** BNG applies combinatorial factors automatically.
For `L(r,r) + R(l) -> L(r!1,r).R(l!1) kf`, a free L has 2 equivalent r
sites, so BNG generates a factor of 2 on the L side. The ODE must include
this factor: `dm/dt = 2*kf*L_free*S - ...`

Three ligand doses are tested: below optimal, optimal (KC ≈ 1),
and above optimal (prozone regime).

In [ ]:
import subprocess, os, glob
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt

os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')

NA = 6.02214076e23; V_cell = 1e-9; R_per_cell = 3e5; f = 0.01
V_sim = V_cell * f; RT = R_per_cell * f; ST = 2 * RT
kon = 1e6; koff = 0.01; KxRT = 5
kf = kon / (NA * V_sim); kxf = KxRT / RT * koff
K1 = kf / koff; K2 = kxf / koff

LT_values = [3e5, 3e6, 3e7]
LT_labels = ['below', 'optimal', 'above']
t_end = 3000; n_steps = 300

print(f'RT={RT:.0f}, ST={ST:.0f}')
print(f'K1={K1:.4e}, K2={K2:.4e}')
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f
    KC = 2 * K1 * LT_sim  # KC = H*A = 2*Ka*C
    print(f'  {label}: LT/cell={LT_val:.0e}, LT_sim={LT_sim:.0f}, KC={KC:.2f}')

## Kinetic ODEs (molecule counts)

$$\frac{dm}{dt} = 2 k_f L_{\text{free}} S - k_{\text{off}} m - k_{xf} m S + 2 k_{\text{off}} M$$
$$\frac{dM}{dt} = k_{xf} m S - 2 k_{\text{off}} M$$

The factor of 2 in the capture term accounts for the bivalent ligand
having 2 identical binding sites (BNG generates `2*kf` per free L
per free R site). Conservation: $S = S_0 - m - 2M$, $L = L_T - m - M$.

In [ ]:
def odes_no_rings(t, y, LT_sim, kf_r, kxf_r, koff_r, ST_r):
    m, M = y
    S = max(ST_r - m - 2*M, 0)
    L = max(LT_sim - m - M, 0)
    # Factor of 2: free bivalent ligand has 2 identical r sites
    dm = 2*kf_r*L*S - koff_r*m - kxf_r*m*S + 2*koff_r*M
    dM = kxf_r*m*S - 2*koff_r*M
    return [dm, dM]

ode_results = {}
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f
    sol = solve_ivp(odes_no_rings, (0, t_end), [0, 0],
                    args=(LT_sim, kf, kxf, koff, ST),
                    method='LSODA', rtol=1e-8, atol=1e-10,
                    t_eval=np.linspace(0, t_end, n_steps+1))
    m, M = sol.y[0], sol.y[1]
    Bonds = m + 2*M
    S = ST - m - 2*M
    ode_results[label] = dict(t=sol.t, m=m, M=M, Bonds=Bonds, S=S,
                              LT_sim=LT_sim)
    print(f'{label}: m_eq={m[-1]:.1f}, M_eq={M[-1]:.1f}, '
          f'Bonds_eq={Bonds[-1]:.1f}, S_eq={S[-1]:.1f}')

## Exact Per-Site Equilibrium

At equilibrium: $m = 2 K_1 L S$, $M = K_1 K_2 L S^2$.
Solve $S = S_T - m - 2M$ with $L = L_T/(1 + 2 K_1 S + K_1 K_2 S^2)$.

In [ ]:
eq = {}
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f
    def f_eq(S_val, LT_sim=LT_sim):
        L = LT_sim / (1 + 2*K1*S_val + K1*K2*S_val**2)
        m_val = 2*K1*L*S_val
        M_val = K1*K2*L*S_val**2
        return S_val - (ST - m_val - 2*M_val)
    S_eq = brentq(f_eq, 1e-10, ST)
    L_eq = LT_sim / (1 + 2*K1*S_eq + K1*K2*S_eq**2)
    m_eq = 2*K1*L_eq*S_eq
    M_eq = K1*K2*L_eq*S_eq**2
    Bonds_eq = m_eq + 2*M_eq
    eq[label] = dict(S=S_eq, m=m_eq, M=M_eq, Bonds=Bonds_eq)
    print(f'{label}: m_eq={m_eq:.1f}, M_eq={M_eq:.1f}, '
          f'Bonds_eq={Bonds_eq:.1f}, S_eq={S_eq:.1f}')

## Run NFsim at Three Ligand Doses

In [ ]:
bngl_file = 'symmetric_bivalent_hapten_receptor_crosslinking_dembo1978.bngl'
with open(bngl_file) as fh:
    bngl_template = fh.read()

nf = {}
f_overrides = {'below': 0.01, 'optimal': 0.003, 'above': 0.002}
for LT_val, label in zip(LT_values, LT_labels):
    f_run = f_overrides[label]
    tmp = bngl_template.replace('LT_per_cell  3e6',
                                f'LT_per_cell  {LT_val:.0e}')
    if f_run != f:
        tmp = tmp.replace('f  0.01  # dimensionless',
                          f'f  {f_run}  # dimensionless')
    tmp = tmp.replace('suffix=>"nfr"', f'suffix=>"nfr_{label}"')
    tmp_file = f'tmp_{label}.bngl'
    with open(tmp_file, 'w') as fh: fh.write(tmp)
    RT_run = R_per_cell * f_run
    LT_run = LT_val * f_run
    print(f'Running {label} (f={f_run}, RT={RT_run:.0f}, '
          f'LT={LT_run:.0f})...', end=' ', flush=True)
    subprocess.run(['bionetgen', 'run', '-i', tmp_file],
                   capture_output=True, text=True, timeout=600)
    gdat = f'tmp_{label}_nfr_{label}.gdat'
    if os.path.exists(gdat):
        d = np.loadtxt(gdat, comments='#')
        nf[label] = dict(t=d[:,0], Free_L=d[:,1], Free_R=d[:,2],
                         Bonds=d[:,3], Free_L_sites=d[:,4],
                         Free_R_sites=d[:,5],
                         LT_sim=LT_run, RT_run=RT_run,
                         f_run=f_run)
        print(f'{len(d)} pts, Bonds~{d[-60:,3].mean():.0f}')
    else: print('FAILED')
for fn in glob.glob('tmp_*'): os.remove(fn)

## Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colors = {'below': 'C0', 'optimal': 'C1', 'above': 'C2'}

for col, label in enumerate(LT_labels):
    if label not in nf:
        for row in range(2):
            axes[row, col].text(0.5, 0.5, 'No data',
                transform=axes[row, col].transAxes, ha='center')
        continue
    ode = ode_results[label]; r = nf[label]
    RT_r = r['RT_run']; ST_r = 2 * RT_r

    ax = axes[0, col]
    ax.axhline(eq[label]['Bonds']/ST, color='r', ls='--', lw=1.5,
               label='Exact equil.')
    ax.plot(ode['t'], ode['Bonds']/ST, 'k-', lw=2, label='ODE')
    ax.plot(r['t'], r['Bonds']/ST_r, '-',
            color=colors[label], lw=1, alpha=0.7, label='NFsim')
    ax.set_ylabel('Bonds / ST' if col == 0 else '')
    ax.set_title(f'{label} (LT/cell={r["LT_sim"]/r["f_run"]:.0e})',
                 fontsize=10)
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.axhline(eq[label]['S']/ST, color='r', ls='--', lw=1.5,
               label='Exact equil.')
    ax.plot(ode['t'], ode['S']/ST, 'k-', lw=2, label='ODE')
    ax.plot(r['t'], r['Free_R_sites']/ST_r, '-',
            color=colors[label], lw=1, alpha=0.7, label='NFsim')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Free R sites / ST' if col == 0 else '')
    ax.legend(fontsize=8)

fig.suptitle('Dembo/Perelson ODEs vs NFsim (no rings)\n'
             'Red dashed = exact equil., black = ODE, colored = NFsim',
             fontsize=12)
fig.tight_layout()
fig.savefig('verify_dembo1978.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved verify_dembo1978.png')

## Error Summary

In [ ]:
print('=== Quantitative comparison (equilibrium, normalized) ===')
print(f'{"":>10s} {"Bonds/ST":>16s} {"S/ST":>16s}')
print(f'{"Dose":>10s} {"ODE":>7s} {"NFsim":>7s} '
      f'{"ODE":>7s} {"NFsim":>7s}')
print('-' * 50)
max_rel_err = 0
for label in LT_labels:
    if label not in nf: continue
    ode = ode_results[label]; r = nf[label]; n_eq = 60
    RT_r = r['RT_run']; ST_r = 2 * RT_r
    B_nf = r['Bonds'][-n_eq:].mean() / ST_r
    S_nf = r['Free_R_sites'][-n_eq:].mean() / ST_r
    B_ode = ode['Bonds'][-1] / ST
    S_ode = ode['S'][-1] / ST
    err_B = abs(B_nf - B_ode) / max(B_ode, 1e-10)
    err_S = abs(S_nf - S_ode) / max(S_ode, 1e-10)
    max_rel_err = max(max_rel_err, err_B, err_S)
    print(f'{label:>10s} {B_ode:7.4f} {B_nf:7.4f} '
          f'{S_ode:7.4f} {S_nf:7.4f}')

print(f'\nMax relative error: {max_rel_err:.4f}')
if max_rel_err < 0.10:
    print('PASS: NFsim agrees with Dembo/Perelson theory (no rings).')
    print('Note: ESA approximation; ~5% error expected without rings.')
else:
    print('FAIL: Significant disagreement detected.')